# Lab CNN — 02: Preprocesamiento y partición del dataset

Este notebook prepara el dataset para el entrenamiento de la CNN. A partir de las imágenes crudas en `data/female/` y `data/male/` se realiza la siguiente cadena de procesamiento:

1. **Carga** de todas las imágenes en espacio de color RGB.
2. **Redimensionamiento** uniforme a 128 × 128 píxeles.
3. **Normalización** de valores al rango [0, 1].
4. **Partición** en conjuntos train / val / test (70 / 15 / 15 %) con semilla fija.
5. **Guardado** de los arrays resultantes en `outputs/data_splits.npz` para reutilización.

**Clases:** `0 → female`, `1 → male`

## 0. Imports y configuración de rutas

In [1]:
from pathlib import Path

import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# Este notebook vive en scripts/, sube un nivel para llegar a la raíz del proyecto
BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data"
OUT_DIR  = BASE_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

CLASSES    = {"female": 0, "male": 1}
EXTENSIONS = {".jpg", ".jpeg", ".png"}
IMG_SIZE   = (128, 128)
SEED       = 42

print("Rutas configuradas:")
for cls in CLASSES:
    p = DATA_DIR / cls
    print(f"  {cls}: {p.relative_to(BASE_DIR)}  →  existe: {p.exists()}")

Rutas configuradas:
  female: data/female  →  existe: True
  male: data/male  →  existe: True


## 1. Carga de imágenes

Se recorren las carpetas de cada clase y se leen todas las imágenes en formato RGB. Las etiquetas se codifican como enteros: `0 = female`, `1 = male`.

> Esta celda puede tardar varios minutos dependiendo del tamaño del dataset.

In [2]:
images = []      # imágenes 128×128 normalizadas, dtype float32 (nunca se guardan en tamaño original)
labels_raw = []  # etiquetas enteras
errores    = 0

for cls_name, cls_label in CLASSES.items():
    folder = DATA_DIR / cls_name
    paths  = [p for p in folder.iterdir() if p.suffix.lower() in EXTENSIONS]
    print(f"Cargando '{cls_name}': {len(paths)} archivos encontrados...", end=" ")
    for p in paths:
        img = cv2.imread(str(p))
        if img is None:
            errores += 1
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        interp = cv2.INTER_AREA if (h > IMG_SIZE[0] or w > IMG_SIZE[1]) else cv2.INTER_LINEAR
        img = cv2.resize(img, IMG_SIZE, interpolation=interp)
        images.append((img / 255.0).astype(np.float32))
        labels_raw.append(cls_label)
    print("OK")

X = np.stack(images)
y = np.array(labels_raw, dtype=np.int32)
del images, labels_raw  # liberar listas auxiliares

print(f"\nImágenes cargadas : {len(X)}")
print(f"Errores de lectura: {errores}")
print(f"Shape de X        : {X.shape}  (N, H, W, C)")
print(f"Dtype de X        : {X.dtype}  |  Rango: [{X.min():.4f}, {X.max():.4f}]")
print(f"Shape de y        : {y.shape}  |  Valores únicos: {np.unique(y)}")

Cargando 'female': 2698 archivos encontrados... 

OK
Cargando 'male': 2720 archivos encontrados... 

OK



Imágenes cargadas : 5418
Errores de lectura: 0
Shape de X        : (5418, 128, 128, 3)  (N, H, W, C)
Dtype de X        : float32  |  Rango: [0.0000, 1.0000]
Shape de y        : (5418,)  |  Valores únicos: [0 1]


## 2. Redimensionamiento a 128 × 128 px

Todas las imágenes se redimensionan a 128 × 128 píxeles — tamaño suficiente para capturar rasgos faciales con una CNN desde cero y que mantiene el consumo de RAM en ~1 GB (vs. ~3.3 GB con 224 × 224). El lab indica 224 × 224 como ejemplo; 128 × 128 cumple el requisito de tamaño uniforme con mayor eficiencia. Se usa interpolación `INTER_AREA` para reducción (preserva detalles al escalar hacia abajo) e `INTER_LINEAR` para ampliación.

In [3]:
# Redimensionamiento integrado en la celda de carga — no se requiere procesamiento adicional.
print(f"Shape tras redimensionamiento: {X.shape}")
print(f"Dtype: {X.dtype}  |  Rango de valores: [{X.min():.4f}, {X.max():.4f}]")

Shape tras redimensionamiento: (5418, 128, 128, 3)


Dtype: float32  |  Rango de valores: [0.0000, 1.0000]


## 3. Normalización al rango [0, 1]

Se divide entre 255 para convertir los valores de píxel de `uint8` a `float32` en el intervalo [0, 1]. Este paso es necesario para que el gradiente descenso converja correctamente durante el entrenamiento.

In [4]:
# Normalización integrada en la celda de carga — no se requiere procesamiento adicional.
print(f"Shape de X : {X.shape}")
print(f"Dtype de X : {X.dtype}")
print(f"Rango de X : [{X.min():.4f}, {X.max():.4f}]")
print(f"Shape de y : {y.shape}  |  Valores únicos: {np.unique(y)}")

Shape de X : (5418, 128, 128, 3)
Dtype de X : float32


Rango de X : [0.0000, 1.0000]
Shape de y : (5418,)  |  Valores únicos: [0 1]


## 4. Partición train / val / test (70 / 15 / 15 %)

Se usa un **split estratificado en dos pasos**:

1. Separar el 70 % para entrenamiento y el 30 % restante como *temp*.
2. Dividir *temp* a la mitad para obtener 15 % de validación y 15 % de prueba.

La estratificación (`stratify=y`) garantiza que la proporción de clases se preserve en cada subconjunto. La semilla `random_state=42` asegura reproducibilidad.

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

del X_temp, y_temp  # liberar memoria

## 5. Resumen de la partición

Se muestra el total de muestras por split y la distribución de clases dentro de cada uno.

In [6]:
CLASS_NAMES = {v: k for k, v in CLASSES.items()}  # {0: 'female', 1: 'male'}

splits = {
    "train": (X_train, y_train),
    "val"  : (X_val,   y_val),
    "test" : (X_test,  y_test),
}

total = len(y)
print("=" * 55)
print("  RESUMEN DE PARTICIÓN — LabCNN")
print("=" * 55)
print(f"\nTotal de muestras : {total}")
print(f"Shape por imagen  : {X_train.shape[1:]}")
print()

for split_name, (X_s, y_s) in splits.items():
    pct = len(y_s) / total * 100
    print(f"  [{split_name:>5}]  {len(y_s):>5} muestras  ({pct:.1f}%)")
    for label, cls_name in CLASS_NAMES.items():
        n = int((y_s == label).sum())
        print(f"           {cls_name:<8}: {n:>5}  ({n / len(y_s) * 100:.1f}%)")
    print()

print("=" * 55)

  RESUMEN DE PARTICIÓN — LabCNN

Total de muestras : 5418
Shape por imagen  : (128, 128, 3)

  [train]   3792 muestras  (70.0%)
           female  :  1888  (49.8%)
           male    :  1904  (50.2%)

  [  val]    813 muestras  (15.0%)
           female  :   405  (49.8%)
           male    :   408  (50.2%)

  [ test]    813 muestras  (15.0%)
           female  :   405  (49.8%)
           male    :   408  (50.2%)



## 6. Guardado de los splits

Los seis arrays (`X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`) se guardan en un único archivo comprimido `outputs/data_splits.npz`. Los ejercicios posteriores pueden cargarlo con `np.load` sin necesidad de reprocesar las imágenes.

In [7]:
out_path = OUT_DIR / "data_splits.npz"

np.savez_compressed(
    out_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
)

size_mb = out_path.stat().st_size / (1024 ** 2)
print(f"Archivo guardado : {out_path.relative_to(BASE_DIR)}")
print(f"Tamaño en disco  : {size_mb:.1f} MB")
print()
print("Para cargar en otros notebooks:")
print("  data = np.load('outputs/data_splits.npz')")
print("  X_train, y_train = data['X_train'], data['y_train']")

Archivo guardado : outputs/data_splits.npz
Tamaño en disco  : 236.5 MB

Para cargar en otros notebooks:
  data = np.load('outputs/data_splits.npz')
  X_train, y_train = data['X_train'], data['y_train']
